# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [27]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [28]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [29]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# BUG FIX: these two lines were previously indented *inside* preprocess_text, after its
# `return` statement -- unreachable dead code, so the reviews were never actually cleaned.
# They now run at the top level, after the function is defined.
df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [30]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [32]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [33]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [34]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [35]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # BUG FIX: zero_grad() was missing. Without it, gradients from every previous epoch
    # keep accumulating on top of each other instead of being recomputed fresh each time,
    # which corrupts training.
    optimizer.zero_grad()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6353
Epoch 10/10 - Loss: 0.3818


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [36]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.785


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [37]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.73      0.86      0.79        96
    Positive       0.85      0.71      0.77       104

    accuracy                           0.79       200
   macro avg       0.79      0.79      0.78       200
weighted avg       0.79      0.79      0.78       200

Confusion Matrix:
[[83 13]
 [30 74]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [38]:
import nltk

# word_tokenize() needs NLTK's 'punkt' tokenizer data downloaded first
nltk.download('punkt')
try:
    nltk.download('punkt_tab')  # required on newer NLTK versions; safe to skip on older ones
except Exception:
    pass

from nltk.tokenize import word_tokenize
import gensim
import numpy as np

# Tokenize the training and testing reviews (X_train / X_test come from the split in Section 3,
# and already contain the cleaned, lowercased text from Section 2)
X_train_tokens = [word_tokenize(review) for review in X_train]
X_test_tokens = [word_tokenize(review) for review in X_test]

print("Example tokenized training review:", X_train_tokens[0])

Example tokenized training review: ['the', 'worst', 'was', 'the', 'salmon', 'sashimi']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Train a Word2Vec model **only on the training reviews** (never on the test set — that would leak test-set information into the model).

In [39]:
# Train a Word2Vec model on the training tokens with the required hyperparameters
w2v_model = gensim.models.Word2Vec(
    X_train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1
)

print("Word2Vec vocabulary size:", len(w2v_model.wv.key_to_index))

Word2Vec vocabulary size: 1800


**Convert each review into a single vector.** Word2Vec gives us one vector *per word*, but our neural network needs one fixed-size vector *per review*. The standard approach is to average the vectors of every word in the review that the model knows (words never seen during training — which can only happen in the test set, since `min_count=1` means every training word is kept — are simply skipped; a review with no known words at all gets an all-zero vector).

In [40]:
def review_to_vector(tokens, model, vector_size=200):
    # Collect the Word2Vec vector for every token that exists in the model's vocabulary
    word_vectors = [model.wv[word] for word in tokens if word in model.wv]

    if len(word_vectors) == 0:
        # No known words in this review -- fall back to a zero vector
        return np.zeros(vector_size)

    # Average all the word vectors into one review-level vector
    return np.mean(word_vectors, axis=0)

X_train_w2v = np.array([review_to_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v = np.array([review_to_vector(tokens, w2v_model) for tokens in X_test_tokens])

print("Train Word2Vec feature matrix shape:", X_train_w2v.shape)
print("Test Word2Vec feature matrix shape:", X_test_w2v.shape)

Train Word2Vec feature matrix shape: (800, 200)
Test Word2Vec feature matrix shape: (200, 200)


**Convert the resulting vectors into PyTorch tensors.**

In [41]:
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)

# y_train_tensor already exists from Section 5; we also need a y_test_tensor for Task 2's evaluation
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

print("X_train_w2v_tensor shape:", X_train_w2v_tensor.shape)
print("X_test_w2v_tensor shape:", X_test_w2v_tensor.shape)

X_train_w2v_tensor shape: torch.Size([800, 200])
X_test_w2v_tensor shape: torch.Size([200, 200])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [42]:
input_size_w2v = X_train_w2v_tensor.shape[1]  # 200 (the Word2Vec vector_size)

model2 = nn.Sequential(
    nn.Linear(input_size_w2v, 128),   # Input layer -> 128 neurons
    nn.ReLU(),

    nn.Linear(128, 64),               # Hidden layer -> 64 neurons
    nn.ReLU(),

    nn.Linear(64, 32),                # Hidden layer -> 32 neurons
    nn.ReLU(),

    nn.Linear(32, 1)                  # Output layer -> 1 neuron (no activation: BCEWithLogitsLoss applies sigmoid internally)
)

print(model2)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


**Loss function and optimizer** — same choice as the first model (`BCEWithLogitsLoss` + `Adam`), but this time with the learning rate the task specifies: `lr = 0.001`.

In [43]:
loss_function2 = nn.BCEWithLogitsLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)

**Train for 15 epochs**, following the same forward pass -> loss -> backpropagation -> weight update cycle as the first model.

In [44]:
epochs2 = 15

for epoch in range(epochs2):
    model2.train()

    optimizer2.zero_grad()   # reset gradients before each forward pass

    # 1. Forward pass
    outputs2 = model2(X_train_w2v_tensor)

    # 2. Calculate loss
    loss2 = loss_function2(outputs2, y_train_tensor)

    # 3. Backpropagation
    loss2.backward()

    # 4. Update model weights
    optimizer2.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs2} - Loss: {loss2.item():.4f}")

Epoch 5/15 - Loss: 0.6934
Epoch 10/15 - Loss: 0.6933
Epoch 15/15 - Loss: 0.6932


**Calculate the test accuracy** of this Word2Vec-based model, the same way as the TF-IDF model above.

In [45]:
model2.eval()

with torch.no_grad():
    test_outputs2 = model2(X_test_w2v_tensor)
    test_probabilities2 = torch.sigmoid(test_outputs2)
    predictions2 = (test_probabilities2 >= 0.5).int().numpy().flatten()

accuracy2 = accuracy_score(y_test, predictions2)
print(f"Test Accuracy (Word2Vec + Deep NN): {accuracy2:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, predictions2, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions2))

Test Accuracy (Word2Vec + Deep NN): 0.480

Classification Report:
              precision    recall  f1-score   support

    Negative       0.48      1.00      0.65        96
    Positive       0.00      0.00      0.00       104

    accuracy                           0.48       200
   macro avg       0.24      0.50      0.32       200
weighted avg       0.23      0.48      0.31       200

Confusion Matrix:
[[ 96   0]
 [104   0]]


d:\ANACONDA\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\ANACONDA\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\ANACONDA\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Summary

- **Base model:** TF-IDF features -> a 3-layer network (64 -> 32 -> 1), trained for 10 epochs.
- **Task 1:** Reviews tokenized, a Word2Vec (Skip-gram) model trained on the *training* tokens only (`vector_size=200`, `window=5`, `min_count=1`, `sg=1`), and each review converted into a single fixed-size vector by averaging its words' Word2Vec vectors.
- **Task 2:** A deeper network (128 -> 64 -> 32 -> 1) trained on those Word2Vec vectors for 15 epochs at `lr=0.001`, evaluated with the same accuracy/classification-report/confusion-matrix workflow as the TF-IDF model — so the two approaches can be compared directly on the printed test accuracy.

Two real bugs found in the original TF-IDF/base-model code were also fixed along the way: the text-cleaning step in Section 2 was previously unreachable dead code (wrong indentation placed it after the function's `return`), and the training loop in Section 8 was missing `optimizer.zero_grad()`, which would have let gradients wrongly accumulate across epochs.